# ISM-CyberRAG - Sprint 3 Development

**Final sprint** of the ISM CyberRAG pipeline for the Australian Information Security Manual.

Pipeline: Pre-Filter > Embed > Multi-Query Expand > Hybrid Search > Rerank > Rerank Threshold Check > Generate Answer > Evaluate (RAGAS) > Log (ClearML)

Sprint 3 adds multi-query expansion, a two-stage out-of-scope guardrail, a pipeline explorer UI, CI/CD, and deployment.

---

**Team Name:** Studio Builders

**Team Members:**

- Sreekar Reddy Edulapalli (25617806)
- Chandan Sreenivasaiah (25674250)
- Ruben Easo Thomas (25598184)

## 0 - Sprint 3 Objectives

This sprint focuses on three pipeline improvements and production readiness:

1. **Multi-Query Expansion**: For each user question, the LLM generates 3 alternate phrasings. All variants are searched independently and results are merged before reranking, improving recall on ambiguous or narrow queries.

2. **Two-Stage OOS Guardrail**: Stage 1 is a keyword pre-filter that catches obviously off-topic queries before embedding. Stage 2 checks the maximum rerank score against a calibrated threshold after retrieval. Together they prevent the LLM from hallucinating answers to questions outside the ISM.

3. **Deployment and CI/CD**: Package the pipeline as a deployable application with automated testing and continuous integration.

## 1 - Environment Setup

In [ ]:
import os
import sys

cwd = os.getcwd()
if os.path.basename(cwd) == "notebooks":
    project_root = os.path.abspath(os.path.join(cwd, ".."))
else:
    project_root = cwd

if not os.path.exists(os.path.join(project_root, "src", "config.py")):
    raise RuntimeError(f"Could not find project root from cwd: {cwd}")

if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from dotenv import load_dotenv
load_dotenv()
print(f"Project root: {project_root}")

## 2 - ClearML Initialization

In [ ]:
from src.config import CLEARML_PROJECT, CLEARML_TASK
from clearml import Task

task = Task.init(project_name=CLEARML_PROJECT, task_name=CLEARML_TASK, reuse_last_task_id=False)
print(f"ClearML task: {task.id}")

## 3 - Initialize Clients and Models

In [ ]:
from src.supabase_utils import get_supabase_client, count_rows
from src.embeddings import load_embedding_model, embed_texts, embed_query
from src.reranking import load_reranker, rerank
from src.retrieval import hybrid_search, multi_query_retrieve
from src.llm import generate_answer
from src.query_expansion import expand_query
from src.guardrail import pre_filter, rerank_threshold_check, OOS_REFUSAL
from src.config import INITIAL_RETRIEVE_COUNT, RERANK_TOP_K, OOS_RERANK_THRESHOLD

supabase = get_supabase_client()
embed_model = load_embedding_model()
reranker = load_reranker()
chunk_count = count_rows(supabase)
print(f"Supabase chunks: {chunk_count}")
print("Models loaded: embedding + reranker")

## 4 - Multi-Query Expansion

Sprint 3 adds multi-query expansion to improve recall. For each user question, the LLM generates 3 alternate phrasings that approach the topic from different angles. All variants (original + 3 generated) are searched independently, and results are merged and deduplicated before reranking.

In [ ]:
test_questions = [
    "What does the ISM say about multi-factor authentication?",
    "How should I handle cryptographic key management?",
    "What are the guidelines for email security?",
]

for q in test_questions:
    variants = expand_query(q)
    print(f"\nOriginal: {q}")
    print(f"Variants ({len(variants)} total):")
    for i, v in enumerate(variants):
        print(f"  [{i}] {v}")

## 5 - OOS Guardrail Calibration

Sprint 3 introduces a two-stage guardrail for out-of-scope (OOS) questions. Stage 1 is a keyword pre-filter that catches obviously off-topic queries before embedding. Stage 2 checks the maximum rerank score against a threshold after retrieval.

This section analyses Sprint 2 rerank scores to calibrate the threshold.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json

# Load sprint 2 results and eval dataset for categories
s2_results = pd.read_csv(os.path.join(project_root, "evaluations", "sprint-2", "sprint2_eval_results.csv"))
with open(os.path.join(project_root, "evaluations", "eval_questions.json")) as f:
    eval_data = json.load(f)
categories = [item.get("category", "unknown") for item in eval_data]
if len(categories) == len(s2_results):
    s2_results["category"] = categories

# Analyse rerank scores by category
if "max_rerank_score" in s2_results.columns:
    inscope = s2_results[s2_results["category"] != "out_of_scope"]["max_rerank_score"]
    oos = s2_results[s2_results["category"] == "out_of_scope"]["max_rerank_score"]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(inscope, bins=20, alpha=0.7, label=f"In-scope (n={len(inscope)})", color="#1565c0")
    ax.hist(oos, bins=20, alpha=0.7, label=f"OOS (n={len(oos)})", color="#c62828")
    ax.axvline(x=OOS_RERANK_THRESHOLD, color="black", linestyle="--", label=f"Threshold = {OOS_RERANK_THRESHOLD}")
    ax.set_xlabel("Max Rerank Score")
    ax.set_ylabel("Count")
    ax.set_title("Rerank Score Distribution: In-Scope vs OOS (Sprint 2 Data)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, "evaluations", "sprint-3", "sprint3_oos_threshold_calibration.png"), dpi=150)
    plt.show()
    
    print(f"In-scope mean: {inscope.mean():.3f}, min: {inscope.min():.3f}")
    print(f"OOS mean: {oos.mean():.3f}, max: {oos.max():.3f}")
    print(f"Current threshold: {OOS_RERANK_THRESHOLD}")
else:
    print("max_rerank_score column not found in sprint 2 results")

In [ ]:
test_oos = [
    "What is the best pizza recipe?",
    "Who won the NBA finals?",
    "Write me a poem about clouds",
    "How should I configure my home WiFi router?",
    "What does ISM-0974 say about authentication?",
]

print("Pre-filter tests:")
for q in test_oos:
    passed, reason = pre_filter(q)
    status = "PASS" if passed else "BLOCK"
    print(f"  [{status}] ({reason:12s}) {q}")

## 6 - End-to-End Pipeline (Sprint 3)

The full Sprint 3 pipeline: pre-filter, embed, expand, multi-query hybrid search, rerank, rerank threshold check, generate.

In [ ]:
def retrieve_sprint3(question: str, include_metadata: bool = False):
    """Full Sprint 3 retrieval: pre-filter -> expand -> multi-query search -> rerank -> guardrail."""
    metadata = {
        "guardrail_stage": None,
        "pre_filter_reason": None,
        "query_variants": [],
        "max_rerank_score": 0.0,
    }

    passed, reason = pre_filter(question)
    metadata["pre_filter_reason"] = reason
    if not passed:
        metadata["guardrail_stage"] = "pre_filter"
        return ([], metadata) if include_metadata else []
    
    queries = expand_query(question)
    metadata["query_variants"] = queries
    
    chunks = multi_query_retrieve(
        supabase,
        lambda q: embed_query(embed_model, q),
        queries,
        match_count=INITIAL_RETRIEVE_COUNT,
    )
    
    reranked = rerank(question, chunks, top_k=RERANK_TOP_K)
    passed, max_score = rerank_threshold_check(reranked, OOS_RERANK_THRESHOLD)
    metadata["max_rerank_score"] = float(max_score)
    
    if not passed:
        metadata["guardrail_stage"] = "rerank_threshold"
        return ([], metadata) if include_metadata else []
    
    return (reranked, metadata) if include_metadata else reranked


def retrieve_sprint3_for_eval(question: str):
    return retrieve_sprint3(question, include_metadata=True)


def generate_sprint3(question: str, chunks: list[dict]) -> str:
    if not chunks:
        return OOS_REFUSAL
    return generate_answer(question, chunks)

# Test with a sample query
test_q = "What are the ISM guidelines for managing cryptographic keys?"
chunks = retrieve_sprint3(test_q)
print(f"Query: {test_q}")
print(f"Retrieved {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  [{i+1}] {c.get('control_id', 'N/A'):10s} score={c.get('rerank_score', 0):.3f} | {c['content'][:80]}...")

if chunks:
    answer = generate_answer(test_q, chunks)
    print(f"\nAnswer:\n{answer}")

## 7 - RAGAS Evaluation

Run the same 100-question evaluation dataset through the Sprint 3 pipeline. Direct comparison with Sprint 1 and Sprint 2.

In [ ]:
from src.evaluation import load_eval_dataset, run_ragas_evaluation, compute_ragas_scores, log_metrics_to_clearml
from src.config import EVAL_DATASET_PATH

eval_dataset = load_eval_dataset(EVAL_DATASET_PATH)

print(f"Running Sprint 3 evaluation ({len(eval_dataset)} questions)...")
eval_results = run_ragas_evaluation(
    eval_dataset,
    retrieve_fn=retrieve_sprint3_for_eval,
    generate_fn=generate_sprint3,
)

print("Computing RAGAS scores...")
metrics, results_df = compute_ragas_scores(eval_results)

### 7.1 - Per-Question RAGAS Scores

In [ ]:
display_cols = [c for c in results_df.columns if c != "contexts"]
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
results_df[display_cols].head(20)

### 7.2 - RAGAS Metrics Bar Chart

In [ ]:
metric_names = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_similarity"]
metric_values = [metrics.get(m, 0) for m in metric_names]
targets = [0.78, 0.82, 0.85, 0.91, 0.93]

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(metric_names))
bars = ax.bar(x, metric_values, color="#1565c0", alpha=0.85, label="Sprint 3 Actual")
ax.scatter(x, targets, color="#c62828", s=100, zorder=5, marker="_", linewidths=3, label="Target")

for i, (v, t) in enumerate(zip(metric_values, targets)):
    color = "#2e7d32" if v >= t else "#c62828"
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold", color=color)

ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in metric_names])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Sprint 3 RAGAS Metrics vs Targets")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(project_root, "evaluations", "sprint-3", "sprint3_ragas_metrics.png"), dpi=150)
plt.show()

### 7.3 - Latency Metrics

In [ ]:
latency_cols = ["retrieval_time_s", "generation_time_s", "total_time_s"]
available_cols = [c for c in latency_cols if c in results_df.columns]

if available_cols:
    print("Latency Statistics:")
    print(results_df[available_cols].describe().round(4))
    
    fig, ax = plt.subplots(figsize=(10, 5))
    for col in available_cols:
        ax.hist(results_df[col], bins=20, alpha=0.6, label=col.replace("_", " ").title())
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Count")
    ax.set_title("Sprint 3 Latency Distribution")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, "evaluations", "sprint-3", "sprint3_latency.png"), dpi=150)
    plt.show()

### 7.4 - Scores by Question Category

In [ ]:
categories = [item.get("category", "unknown") for item in eval_dataset]
if len(categories) == len(results_df):
    results_df["category"] = categories

category_means = results_df.groupby("category")[metric_names].mean()
print("Per-category average scores:")
print(category_means.round(4).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
category_means.plot(kind="bar", ax=ax, width=0.8)
ax.set_ylabel("Score")
ax.set_title("Sprint 3 RAGAS Metrics by Question Category")
ax.legend(loc="lower right", fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(project_root, "evaluations", "sprint-3", "sprint3_category_breakdown.png"), dpi=150)
plt.show()

### 7.5 - In-Scope vs Out-of-Scope Analysis

In [ ]:
import numpy as np

if "category" in results_df.columns:
    inscope_mask = results_df["category"] != "out_of_scope"
    oos_mask = results_df["category"] == "out_of_scope"
    
    all_means = results_df[metric_names].mean()
    inscope_means = results_df.loc[inscope_mask, metric_names].mean()
    oos_means = results_df.loc[oos_mask, metric_names].mean()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(metric_names))
    width = 0.25
    
    ax.bar(x - width, all_means, width, label="All (100)", color="#1565c0", alpha=0.85)
    ax.bar(x, inscope_means, width, label="In-Scope (90)", color="#2e7d32", alpha=0.85)
    ax.bar(x + width, oos_means, width, label="OOS (10)", color="#c62828", alpha=0.85)
    
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("_", "\n") for m in metric_names])
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score")
    ax.set_title("Sprint 3: All vs In-Scope vs OOS")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, "evaluations", "sprint-3", "sprint3_inscope_vs_oos.png"), dpi=150)
    plt.show()

## 8 - Three-Sprint Comparison

Compare Sprint 1, Sprint 2, and Sprint 3 RAGAS metrics side by side.

In [ ]:
sprint1_scores = {
    "faithfulness": 0.6834,
    "answer_relevancy": 0.7216,
    "context_precision": 0.7885,
    "context_recall": 0.8224,
    "answer_similarity": None,
}

sprint2_scores = {
    "faithfulness": 0.7341,
    "answer_relevancy": 0.7678,
    "context_precision": 0.8598,
    "context_recall": 0.8659,
    "answer_similarity": 0.9057,
}

sprint3_scores = {m: metrics.get(m, 0) for m in metric_names}
targets = {"faithfulness": 0.78, "answer_relevancy": 0.82, "context_precision": 0.85, "context_recall": 0.91, "answer_similarity": 0.93}

comparison_data = []
for m in metric_names:
    s1 = sprint1_scores.get(m)
    s2 = sprint2_scores.get(m)
    s3 = sprint3_scores.get(m)
    t = targets.get(m)
    comparison_data.append({
        "Metric": m,
        "Sprint 1": f"{s1:.4f}" if s1 is not None else "N/A",
        "Sprint 2": f"{s2:.4f}" if s2 is not None else "N/A",
        "Sprint 3": f"{s3:.4f}" if s3 is not None else "N/A",
        "Target": f"> {t}",
        "Target Met": s3 >= t if s3 is not None and t is not None else False,
    })

comparison_df = pd.DataFrame(comparison_data)
print("Three-Sprint Comparison:")
print(comparison_df.to_string(index=False))

# Save comparison CSV
comparison_df.to_csv(os.path.join(project_root, "evaluations", "sprint-3", "sprint3_vs_sprint2_vs_sprint1.csv"), index=False)

# Comparison bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(metric_names))
width = 0.2

s1_vals = [sprint1_scores.get(m, 0) or 0 for m in metric_names]
s2_vals = [sprint2_scores.get(m, 0) or 0 for m in metric_names]
s3_vals = [sprint3_scores.get(m, 0) for m in metric_names]
target_vals = [targets.get(m, 0) for m in metric_names]

ax.bar(x - width, s1_vals, width, label="Sprint 1", color="#90caf9", alpha=0.9)
ax.bar(x, s2_vals, width, label="Sprint 2", color="#1565c0", alpha=0.9)
ax.bar(x + width, s3_vals, width, label="Sprint 3", color="#0d47a1", alpha=0.9)
ax.scatter(x + width, target_vals, color="#c62828", s=100, zorder=5, marker="_", linewidths=3, label="Sprint 3 Target")

ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in metric_names])
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("ISM CyberRAG: Sprint 1 vs Sprint 2 vs Sprint 3")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(project_root, "evaluations", "sprint-3", "sprint3_comparison.png"), dpi=150)
plt.show()

## 9 - ClearML Logging and Save Results

In [ ]:
from src.config import EVAL_LLM_PROVIDER, EVAL_LLM_MODEL
from src.evaluation import log_metrics_to_clearml
from src.config import (
    EMBEDDING_MODEL_NAME, RERANKER_MODEL, LLM_MODEL_NAME,
    INITIAL_RETRIEVE_COUNT, RERANK_TOP_K, RRF_K,
    MULTI_QUERY_COUNT, MULTI_QUERY_ENABLED, OOS_RERANK_THRESHOLD,
)

params = {
    "sprint": 3,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "reranker_model": RERANKER_MODEL,
    "llm_model": LLM_MODEL_NAME,
    "eval_llm_provider": EVAL_LLM_PROVIDER,
    "eval_llm_model": EVAL_LLM_MODEL,
    "initial_retrieve_count": INITIAL_RETRIEVE_COUNT,
    "rerank_top_k": RERANK_TOP_K,
    "rrf_k": RRF_K,
    "multi_query_count": MULTI_QUERY_COUNT,
    "multi_query_enabled": MULTI_QUERY_ENABLED,
    "oos_rerank_threshold": OOS_RERANK_THRESHOLD,
    "chunk_count": chunk_count,
    "eval_questions": len(eval_dataset),
}

log_metrics_to_clearml(metrics, params=params, results_df=results_df, eval_results=eval_results)

### 9.1 - Save Evaluation Results

In [ ]:
results_path = os.path.join(project_root, "evaluations", "sprint-3", "sprint3_eval_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Saved: {results_path} ({len(results_df)} rows)")

---
## 10 - Close ClearML Task

In [ ]:
from clearml import Task
current = Task.current_task()
if current:
    current.close()
    print(f"ClearML task {current.id} closed.")